# UMAP Explorer — Interactive Region Characterization

1. Run Cell 1 to load data
2. Run Cell 2 to show the interactive UMAP plot
3. **Use the lasso or box select** tool (toolbar top-right) to select a region
4. Run Cell 3 to see feature stats for the selected region
5. Copy the stats table and paste into Claude for discussion

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

DATA_DIR = Path("../outputs/data")

umap_df = pd.read_parquet(DATA_DIR / "umap_coordinates.parquet")
feat_df = pd.read_parquet(DATA_DIR / "event_features.parquet")
df = umap_df.merge(feat_df, on=["event_id", "mooring", "detection_band"], how="left")

STAT_FEATURES = [
    "peak_freq_hz", "duration_s", "snr", "bandwidth_hz",
    "spectral_slope", "freq_modulation", "spectral_centroid_hz",
    "peak_power_db", "decay_time_s", "rise_time_s",
]

saved_regions = {}

print(f"Loaded {len(df):,} events")
print(f"Bands: {df['detection_band'].value_counts().to_dict()}")

Loaded 297,170 events
Bands: {'mid': 132494, 'low': 84698, 'high': 79978}


In [2]:
# Cell 2: Interactive UMAP with plotly lasso/box selection
# Change BAND to "low", "mid", or "high"
BAND = "low"

import plotly.graph_objects as go

band_df = df[df["detection_band"] == BAND].copy()

MAX_PTS = 40000
if len(band_df) > MAX_PTS:
    band_df = band_df.sample(MAX_PTS, random_state=42).reset_index(drop=True)
    print(f"Subsampled to {MAX_PTS:,} points")
else:
    band_df = band_df.reset_index(drop=True)

labels = band_df["cluster_id_numeric"].values
cluster_ids = sorted(set(labels) - {-1})

fig = go.FigureWidget()

noise_mask = labels == -1
if noise_mask.any():
    fig.add_trace(go.Scattergl(
        x=band_df.loc[noise_mask, "umap_1"],
        y=band_df.loc[noise_mask, "umap_2"],
        mode="markers",
        marker=dict(size=2, color="lightgray", opacity=0.3),
        name="noise",
        customdata=band_df.loc[noise_mask].index,
    ))

for cid in cluster_ids:
    mask = labels == cid
    fig.add_trace(go.Scattergl(
        x=band_df.loc[mask, "umap_1"],
        y=band_df.loc[mask, "umap_2"],
        mode="markers",
        marker=dict(size=2, opacity=0.4),
        name=f"{BAND}_{cid} ({mask.sum():,})",
        customdata=band_df.loc[mask].index,
    ))

fig.update_layout(
    title=f"{BAND.upper()} band — {len(band_df):,} events — use lasso/box select",
    xaxis_title="UMAP 1",
    yaxis_title="UMAP 2",
    dragmode="lasso",
    width=900,
    height=700,
    legend=dict(font=dict(size=10)),
)

fig

Subsampled to 40,000 points


FigureWidget({
    'data': [{'customdata': {'bdata': ('AgAAAAMAAAAFAAAADgAAABQAAAAlAA' ... 'AADZwAACCcAAAonAAAKZwAADGcAAA='),
                             'dtype': 'i4'},
              'marker': {'color': 'lightgray', 'opacity': 0.3, 'size': 2},
              'mode': 'markers',
              'name': 'noise',
              'type': 'scattergl',
              'uid': '15171c21-8c5b-4da8-b51c-00b965aa7723',
              'x': {'bdata': ('TBO5QFT3D0FjPpVAFqgUQQyUxECnXl' ... 'ZBQz4QQSVUXUFOgalAiJhvQULHLEE='),
                    'dtype': 'f4'},
              'y': {'bdata': ('DDTHP8K6CMDijxxA/v4WQUPVST5vA4' ... 'RBTFX+vxQQ5r6bEQVB3nVtwNOSJUE='),
                    'dtype': 'f4'}},
             {'customdata': {'bdata': ('EgAAAB8AAAAxAAAAMwAAADYAAAA3AA' ... 'AA+JsAAPubAAAenAAAIpwAADycAAA='),
                             'dtype': 'i4'},
              'marker': {'opacity': 0.4, 'size': 2},
              'mode': 'markers',
              'name': 'low_0 (2,165)',
              'type': 'scattergl',
 

In [5]:
# Cell 3: CHARACTERIZE SELECTION
# Run this after using lasso/box select on the plot above.
# Plotly stores selected point indices — we extract them here.

selected_indices = []
for trace_data in fig.data:
    if trace_data.selectedpoints:
        cd = trace_data.customdata
        selected_indices.extend([cd[i] for i in trace_data.selectedpoints])

if not selected_indices:
    print("No points selected. Use the lasso or box select tool on the plot above first.")
else:
    sel_df = band_df.loc[selected_indices]
    lines = []
    lines.append(f"## UMAP Region")
    lines.append(f"Band: {BAND} | Events: {len(sel_df):,}")
    lines.append("")
    lines.append("| Feature | Min | P5 | Median | Mean | P95 | Max |")
    lines.append("|---------|-----|-----|--------|------|-----|-----|")
    for feat in STAT_FEATURES:
        vals = sel_df[feat].dropna()
        if len(vals) > 0:
            lines.append(
                f"| {feat} | {vals.min():.2f} | {vals.quantile(0.05):.2f} "
                f"| {vals.median():.2f} | {vals.mean():.2f} "
                f"| {vals.quantile(0.95):.2f} | {vals.max():.2f} |"
            )

    lines.append("")
    lines.append("Cluster breakdown:")
    for cid, count in sel_df["cluster_id"].value_counts().items():
        lines.append(f"  {cid}: {count:,}")

    summary = "\n".join(lines)
    display(Markdown(summary))
    print("\n--- Copy below this line ---")
    print(summary)
    print("--- Copy above this line ---")

No points selected. Use the lasso or box select tool on the plot above first.


In [ ]:
# Cell 4: NAME AND SAVE REGION
# Change REGION_NAME then run this cell.

REGION_NAME = "unnamed"  # <-- change this, e.g. "texas_peninsula"

if not selected_indices:
    print("No selection to save.")
else:
    sel_df = band_df.loc[selected_indices]
    saved_regions[REGION_NAME] = {
        "band": BAND,
        "n_events": len(selected_indices),
        "event_ids": sel_df["event_id"].tolist(),
    }
    print(f"Saved '{REGION_NAME}' ({len(selected_indices):,} events)")
    print(f"All saved: {list(saved_regions.keys())}")

In [ ]:
# Cell 5: EXPORT ALL SAVED REGIONS
import json

if saved_regions:
    outpath = DATA_DIR / "manual_umap_regions.json"
    with open(outpath, "w") as f:
        json.dump(saved_regions, f, indent=2, default=str)
    print(f"Exported {len(saved_regions)} regions to {outpath}")
    for name, info in saved_regions.items():
        print(f"  {name}: {info['n_events']:,} events ({info['band']})")
else:
    print("No regions saved yet.")